In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
# =====================================================================
#  STORE SALES — FINAL WORKING SOLUTION (FIXED)
# =====================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import time
import gc

print("\n" + "="*70)
print("  STORE SALES — FINAL WORKING SOLUTION")
print("="*70 + "\n")

T0 = time.time()

# =====================================================================
# 1. LOAD DATA
# =====================================================================
print("[1/7] Loading data...")

BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")

train = pd.read_csv(BASE/"train.csv", parse_dates=["date"])
test = pd.read_csv(BASE/"test.csv", parse_dates=["date"])
stores = pd.read_csv(BASE/"stores.csv")
oil = pd.read_csv(BASE/"oil.csv", parse_dates=["date"])
holidays = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans = pd.read_csv(BASE/"transactions.csv", parse_dates=["date"])

test_ids = test[["id"]].copy().reset_index(drop=True)

print(f"     Train: {train.shape}")
print(f"     Test:  {test.shape}")

# =====================================================================
# 2. PREPROCESSING
# =====================================================================
print("\n[2/7] Preprocessing...")

train["sales"] = train["sales"].clip(lower=0).astype("float32")
train["log_sales"] = np.log1p(train["sales"])

stores["type_code"] = LabelEncoder().fit_transform(stores["type"])
stores["city_code"] = LabelEncoder().fit_transform(stores["city"])
stores["state_code"] = LabelEncoder().fit_transform(stores["state"])

oil_dates = pd.date_range(oil.date.min(), "2017-08-31", freq="D")
oil = oil.set_index("date").reindex(oil_dates).reset_index()
oil.columns = ["date", "dcoilwtico"]
oil["dcoilwtico"] = oil["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
oil["oil_ma7"] = oil["dcoilwtico"].rolling(7, min_periods=1).mean()
oil["oil_ma30"] = oil["dcoilwtico"].rolling(30, min_periods=1).mean()

hol_national = set(holidays[holidays.locale == "National"]["date"])
hol_df = pd.DataFrame({"date": pd.date_range("2013-01-01", "2017-08-31", freq="D")})
hol_df["is_holiday"] = hol_df.date.isin(hol_national).astype(int)

trans_daily = trans.groupby(["date", "store_nbr"])["transactions"].sum().reset_index()

print("     ✓ Done")

# =====================================================================
# 3. BUILD DATASET
# =====================================================================
print("\n[3/7] Building dataset...")

data = pd.concat([
    train[["date", "store_nbr", "family", "sales", "log_sales", "onpromotion"]].assign(is_test=False),
    test[["date", "store_nbr", "family", "onpromotion"]].assign(sales=np.nan, log_sales=np.nan, is_test=True)
], ignore_index=True)

data = data.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

data = data.merge(stores[["store_nbr", "type_code", "cluster", "city_code", "state_code"]], 
                  on="store_nbr", how="left")
data = data.merge(oil[["date", "dcoilwtico", "oil_ma7", "oil_ma30"]], on="date", how="left")
data = data.merge(hol_df, on="date", how="left")
data = data.merge(trans_daily, on=["date", "store_nbr"], how="left")

data["year"] = data.date.dt.year
data["month"] = data.date.dt.month
data["day"] = data.date.dt.day
data["dow"] = data.date.dt.dayofweek
data["is_wknd"] = (data.dow >= 5).astype(int)
data["is_payday"] = ((data.day == 15) | (data.day >= 28)).astype(int)

le_fam = LabelEncoder().fit(train.family)
data["family_code"] = le_fam.transform(data.family)

print(f"     ✓ Shape: {data.shape}")

# =====================================================================
# 4. LAG FEATURES
# =====================================================================
print("\n[4/7] Creating lags...")

grp = data.groupby(["store_nbr", "family"])["log_sales"]

for lag in [7, 14, 21, 28, 35, 42, 56, 91, 112, 364, 365, 366]:
    data[f"lag_{lag}"] = grp.shift(lag)

for w in [7, 14, 28]:
    data[f"roll_mean_{w}"] = grp.shift(16).rolling(w, min_periods=1).mean()

print(f"     ✓ {data.shape[1]} features")

# =====================================================================
# 5. SPLIT
# =====================================================================
print("\n[5/7] Splitting...")

is_test = data["is_test"].values
test_data = data[is_test].copy()
train_data = data[~is_test].copy()

mask_ok = train_data["lag_112"].notna() & train_data["log_sales"].notna()
train_clean = train_data[mask_ok].copy()

val_date = pd.Timestamp("2017-04-01")
mask_tr = train_clean.date < val_date
mask_va = train_clean.date >= val_date

feature_cols = [c for c in data.columns if c not in 
                ["date", "family", "sales", "log_sales", "is_test", "type", "city", "state"]]

X_tr = train_clean[mask_tr][feature_cols].fillna(0)
y_tr = train_clean[mask_tr]["log_sales"]
X_va = train_clean[mask_va][feature_cols].fillna(0)
y_va = train_clean[mask_va]["log_sales"]
X_te = test_data[feature_cols].fillna(0)

print(f"     Train: {X_tr.shape[0]:,}  Val: {X_va.shape[0]:,}  Test: {X_te.shape[0]:,}")

del data, train_data, train_clean; gc.collect()

# =====================================================================
# 6. TRAIN
# =====================================================================
print("\n[6/7] Training LightGBM...")

params = {
    "objective": "regression_l2",
    "metric": "rmse",
    "num_leaves": 200,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "min_child_samples": 20,
    "verbose": -1,
    "n_jobs": -1,
}

dtrain = lgb.Dataset(X_tr, y_tr)
dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

model = lgb.train(
    params, dtrain,
    num_boost_round=2000,
    valid_sets=[dvalid],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(200),
    ],
)

print(f"     ✓ Best iter: {model.best_iteration}")

# =====================================================================
# 7. PREDICT
# =====================================================================
print("\n[7/7] Validation & prediction...")

va_pred_raw = model.predict(X_va, num_iteration=model.best_iteration)
va_pred = np.expm1(va_pred_raw).clip(0)
va_true = np.expm1(y_va.values).clip(0)

rmsle = np.sqrt(np.mean((np.log1p(va_pred) - np.log1p(va_true))**2))
print(f"     Val RMSLE: {rmsle:.5f}")

te_pred_raw = model.predict(X_te, num_iteration=model.best_iteration)
te_pred = np.expm1(te_pred_raw).clip(0)

print(f"     Test pred: mean={te_pred.mean():.2f} min={te_pred.min():.2f} max={te_pred.max():.2f}")

# =====================================================================
# 8. SAVE
# =====================================================================
print("\n[8/8] Saving submission...")

# FIX: te_pred is already numpy array, no .values needed
sub = test_ids.copy()
sub["sales"] = te_pred
sub = sub.sort_values("id").reset_index(drop=True)

assert len(sub) == 28512, f"Wrong count: {len(sub)}"
assert not sub.sales.isna().any(), "NaN found!"
assert sub.sales.min() >= 0, "Negative sales!"

out = Path("/kaggle/working/submission.csv")
sub.to_csv(out, index=False)

print(f"     ✓ Saved: {out}")
print(f"     Rows: {len(sub):,}")
print(sub[["id", "sales"]].head(10).to_string(index=False))

print(f"\n" + "="*70)
print(f"  ✅ COMPLETE in {int(time.time()-T0)}s")
print(f"  Val RMSLE:  {rmsle:.5f}")
print(f"  Expected:   0.37-0.42 range")
print(f"="*70 + "\n")


  STORE SALES — FINAL WORKING SOLUTION

[1/7] Loading data...
     Train: (3000888, 6)
     Test:  (28512, 5)

[2/7] Preprocessing...
     ✓ Done

[3/7] Building dataset...
     ✓ Shape: (3029400, 23)

[4/7] Creating lags...
     ✓ 38 features

[5/7] Splitting...
     Train: 2,557,170  Val: 244,134  Test: 28,512

[6/7] Training LightGBM...
[200]	valid_0's rmse: 0.41821
[400]	valid_0's rmse: 0.41469
[600]	valid_0's rmse: 0.413482
[800]	valid_0's rmse: 0.413139
[1000]	valid_0's rmse: 0.412688
     ✓ Best iter: 1105

[7/7] Validation & prediction...
     Val RMSLE: 0.41257
     Test pred: mean=30.32 min=0.00 max=1677.24

[8/8] Saving submission...
     ✓ Saved: /kaggle/working/submission.csv
     Rows: 28,512
     id    sales
3000888 0.000000
3000889 0.000000
3000890 0.000000
3000891 0.372656
3000892 0.000000
3000893 0.000000
3000894 0.089084
3000895 0.096709
3000896 0.181497
3000897 0.000000

  ✅ COMPLETE in 179s
  Val RMSLE:  0.41257
  Expected:   0.37-0.42 range

